In [ ]:
# Imports
import sys
from pathlib import Path
sys.path.insert(0, "..") 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from catboost import CatBoostClassifier

from src.constants import (
    TARGET_COL,
    MULTILABEL_COLS,
    CATEGORICAL_COLS,
    CLASS_LABELS,
    RANDOM_STATE,
    CV_SPLITS,
    PROCESSED_DATA_PATH,
    ARTIFACTS_PATH
)

from src.preprocessing import load_processed_dataset, temporal_train_val_test_split

from src.encoders import (
    fit_mlb_encoder,
    fit_ohe_encoder,
    build_mlb_cat_matrix,
    build_mlb_ohe_matrix,
    save_encoders
)

from src.evaluation import (
    evaluate_classification_model,
    fit_catboost_classifier,
    compare_results_table
)

from src.visualisations import (
    plot_calibration_curve,
    plot_feature_importance,
    get_original_feature
)

In [ ]:
# Load preprocessed dataset
df = load_processed_dataset(PROCESSED_DATA_PATH)
df.info()

# Apply and verify the temporal train/test split
df_train, df_val, df_test = temporal_train_val_test_split(df)
print(f"\nTraining rows: {len(df_train)}")
print(f"Validation rows: {len(df_val)}")
print(f"Testing rows: {len(df_test)}")
print(f"\nTraining timeframe:\n {df_train[["year", "quarter"]].drop_duplicates().sort_values(['year', 'quarter']).to_string(index=False)}")
print(f"\nValidation timeframe:\n {df_val[["year", "quarter"]].drop_duplicates().sort_values(['year', 'quarter']).to_string(index=False)}")
print(f"\nTesting timeframe:\n {df_test[["year", "quarter"]].drop_duplicates().sort_values(['year', 'quarter']).to_string(index=False)}")

In [ ]:
print(df_train[TARGET_COL].value_counts(normalize=True))
print(df_val[TARGET_COL].value_counts(normalize=True))
print(df_test[TARGET_COL].value_counts(normalize=True))

In [ ]:
# Fit encoders on the training set 
mlb_encoder = fit_mlb_encoder(df_train, MULTILABEL_COLS)
ohe_encoder = fit_ohe_encoder(df_train, CATEGORICAL_COLS)

# Verify the number of unique labels found for each multilabel column
for col, encoder in mlb_encoder.items():
    print(f"{col}: {len(encoder.classes_)} unique labels found")

In [ ]:
# Build mlb + categorical feature matrices for CatBoost native handling
X_train_cat, cat_feature_indices = build_mlb_cat_matrix(df_train, mlb_encoder)
X_val_cat, _ = build_mlb_cat_matrix(df_val, mlb_encoder)
X_test_cat, _ = build_mlb_cat_matrix(df_test, mlb_encoder)

y_train = df_train[TARGET_COL].reset_index(drop=True)
y_val = df_val[TARGET_COL].reset_index(drop=True)
y_test = df_test[TARGET_COL].reset_index(drop=True)

X_train_val_cat = pd.concat([X_train_cat, X_val_cat], axis=0).reset_index(drop=True)
y_train_val = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)


print(f"Categorical indicies: {cat_feature_indices}")
print(f"Categorical columns: {CATEGORICAL_COLS}")
print(f"X_train_cat shape: {X_train_cat.shape}")
print(f"X_val_cat shape: {X_val_cat.shape}")
print(f"X_test_cat shape: {X_test_cat.shape}")
print(f"\nHead of X_train_cat:\n {X_train_cat.head(3).to_string()}")

assert X_train_cat.isnull().sum().sum() == 0, "Null values found in X_train_cat"
assert X_test_cat.isnull().sum().sum() == 0, "Null values found in X_test_cat"
print("\nNo null values found in mlb + cat feature matrices.")


In [ ]:
# Train MLB + OHE CatBoost model
X_train_ohe = build_mlb_ohe_matrix(df_train, mlb_encoder, ohe_encoder)
X_val_ohe = build_mlb_ohe_matrix(df_val, mlb_encoder, ohe_encoder)
X_test_ohe = build_mlb_ohe_matrix(df_test, mlb_encoder, ohe_encoder)

print(f"X_train_ohe shape: {X_train_ohe.shape}")
print(f"X_val_ohe shape: {X_val_ohe.shape}")
print(f"X_test_ohe shape: {X_test_ohe.shape}")
print(f"\nHead of X_train_ohe:\n {X_train_ohe.head(3).to_string()}")

In [ ]:
mlb_cat_models = {}
mlb_cat_results = {}

mlb_cat_models["none"], mlb_cat_results["none"] = fit_catboost_classifier(
    "CatBoost No Weights Baseline (MLB + Categorical)",
    X_train_cat,
    y_train,
    X_val_cat,
    y_val,
    cat_feature_indices=cat_feature_indices,
    class_weights=None,
)

mlb_cat_models["baseline"], mlb_cat_results["baseline"] = fit_catboost_classifier(
    "CatBoost Balanced Baseline (MLB + Categorical)",
    X_train_cat,
    y_train,
    X_val_cat,
    y_val,
    cat_feature_indices=cat_feature_indices,
    class_weights="Balanced",
)

In [ ]:
mlb_ohe_models = {}
mlb_ohe_results = {}

mlb_ohe_models["none"], mlb_ohe_results["none"] = fit_catboost_classifier(
    "CatBoost No Weights Baseline (MLB + OHE)",
    X_train_ohe,
    y_train,
    X_val_ohe,
    y_val,
    cat_feature_indices=None,
    class_weights=None,
)

mlb_ohe_models["baseline"], mlb_ohe_results["baseline"] = fit_catboost_classifier(
    "CatBoost Balanced Baseline (MLB + OHE)",
    X_train_ohe,
    y_train,
    X_val_ohe,
    y_val,
    cat_feature_indices=None,
    class_weights="Balanced",
)

In [ ]:
cb_encoding_comparison = compare_results_table(
    [mlb_cat_results["none"], mlb_cat_results["baseline"], mlb_ohe_results["none"], mlb_ohe_results["baseline"]],
    print_table=False
)
display(cb_encoding_comparison)

In [ ]:
tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
param_grid = {
    "learning_rate": [0.03, 0.05, 0.07, 0.1],
    "depth": [6, 8, 10, 12],
    "l2_leaf_reg": [1, 3, 5, 7]
}

In [ ]:
bw_tuned_model = CatBoostClassifier(
    iterations=2000,
    loss_function="MultiClass",
    random_state=RANDOM_STATE,
    cat_features=cat_feature_indices,
    early_stopping_rounds=50,
    eval_metric="TotalF1:average=Macro",
    auto_class_weights="Balanced",
    verbose=False
)

bw_grid_result = bw_tuned_model.grid_search(
    param_grid,
    X=X_train_cat,
    y=y_train,
    cv=tscv,
    stratified=False,
    refit=False,
    shuffle=False,
    verbose=False
)

In [ ]:
nw_tuned_model = CatBoostClassifier(
    iterations=2000,
    loss_function="MultiClass",
    random_state=RANDOM_STATE,
    cat_features=cat_feature_indices,
    early_stopping_rounds=50,
    eval_metric="TotalF1:average=Macro",
    auto_class_weights=None,
    verbose=False
)

nw_grid_result = nw_tuned_model.grid_search(
    param_grid,
    X=X_train_cat,
    y=y_train,
    cv=tscv,
    stratified=False,
    refit=False,
    shuffle=False,
    verbose=False
)

In [ ]:
print("\nBest parameters found:")
for key, value in bw_grid_result["params"].items():
    print(f"\t{key}: {value}")

bw_cv_curve = pd.DataFrame(bw_grid_result["cv_results"])
bw_best_iteration = bw_cv_curve["test-TotalF1:average=Macro-mean"].idxmax()
bw_train_f1 = bw_cv_curve.loc[bw_best_iteration, "train-TotalF1:average=Macro-mean"]
bw_test_f1  = bw_cv_curve.loc[bw_best_iteration, "test-TotalF1:average=Macro-mean"]
bw_test_std = bw_cv_curve.loc[bw_best_iteration, "test-TotalF1:average=Macro-std"]

print(f"\nCV diagnostics for best combination:")
print(f"\tBest iteration: {bw_best_iteration} of {len(bw_cv_curve)} total")
print(f"\tTrain F1: {bw_train_f1:.4f}")
print(f"\tTest F1: {bw_test_f1:.4f} (std across folds: {bw_test_std:.4f})")
print(f"\tTrain-test gap: {bw_train_f1 - bw_test_f1:.4f}")

In [ ]:
print("\nBest parameters found:")
for key, value in nw_grid_result["params"].items():
    print(f"\t{key}: {value}")

nw_cv_curve = pd.DataFrame(nw_grid_result["cv_results"])
nw_best_iteration = nw_cv_curve["test-TotalF1:average=Macro-mean"].idxmax()
nw_train_f1 = nw_cv_curve.loc[nw_best_iteration, "train-TotalF1:average=Macro-mean"]
nw_test_f1  = nw_cv_curve.loc[nw_best_iteration, "test-TotalF1:average=Macro-mean"]
nw_test_std = nw_cv_curve.loc[nw_best_iteration, "test-TotalF1:average=Macro-std"]

print(f"\nCV diagnostics for best combination:")
print(f"\tBest iteration: {nw_best_iteration} of {len(nw_cv_curve)} total")
print(f"\tTrain F1: {nw_train_f1:.4f}")
print(f"\tTest F1: {nw_test_f1:.4f} (std across folds: {nw_test_std:.4f})")
print(f"\tTrain-test gap: {nw_train_f1 - nw_test_f1:.4f}")

In [ ]:
balanced_catboost_tuned, balanced_catboost_tuned_results = fit_catboost_classifier(
    "CatBoost Tuned (Balanced MLB + CAT)",
    X_train_cat,
    y_train,
    X_val_cat,
    y_val,
    cat_feature_indices=cat_feature_indices,
    class_weights="Balanced",
    hyperparameters= bw_grid_result["params"]
)

print(f"\nFinal CatBoost model, Best iteration found: {balanced_catboost_tuned.get_best_iteration()}")

In [ ]:
importances = balanced_catboost_tuned.get_feature_importance()
importance_df = pd.DataFrame({
    "feature": X_train_cat.columns,
    "importance": importances,
})
importance_df["original_feature"] = importance_df["feature"].apply(
    lambda c: get_original_feature(c, CATEGORICAL_COLS)
)
importance_grouped = importance_df.groupby("original_feature")["importance"].sum().sort_values(ascending=False)

print("Feature importance (grouped by original feature):")
print(importance_grouped.to_string())

print("\nTop 15 raw features (including individual MLB sub-columns):")
print(importance_df.sort_values("importance", ascending=False).head(15).to_string(index=False))

plot_feature_importance(
    importances,
    list(X_train_cat.columns),
    title="Feature Importance: Final CatBoost Tuned Model",
    categorical_cols=CATEGORICAL_COLS,
    group_by_original_feature=True,
)

In [ ]:
permutation_results = permutation_importance(
    balanced_catboost_tuned,
    X_test_cat,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
    scoring="f1_macro"
)

permutation_importance_df = pd.DataFrame({
    "feature": X_train_cat.columns,
    "importance_mean": permutation_results["importances_mean"],
    "importance_std": permutation_results["importances_std"]
})
permutation_importance_df["original_feature"] = permutation_importance_df["feature"].apply(
    lambda c: get_original_feature(c, CATEGORICAL_COLS)
)
permutation_grouped = permutation_importance_df.groupby("original_feature")["importance_mean"].sum().sort_values(ascending=False)

combined_importance = pd.DataFrame({
    "built_in": importance_grouped,
    "permutation": permutation_grouped
}).sort_values("built_in", ascending=False)

combined_importance_percent = combined_importance.div(combined_importance.sum()).mul(100).round(2)
print("\nCombined feature importance (built-in + permutation, grouped by original feature):")
print(combined_importance_percent.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
combined_importance_percent.iloc[::-1].plot(kind="barh", ax=ax, width=0.75)
ax.set_title("Feature Importance: Built-in vs Permutation", fontweight="bold")
ax.set_xlabel("Percentage of Total Importance")
ax.legend(["Built-in (split frequency)", "Permutation (test F1 drop)"])
plt.tight_layout()
plt.show()

print(f"\nRaw permutation importance (per encoded column, top 15):")
top_perm = permutation_importance_df.sort_values("importance_mean", ascending=False).head(15)
for _, row in top_perm.iterrows():
    print(f"\t{row['feature']:30s}, Mean drop: {row['importance_mean']:+.4f}, Std: {row['importance_std']:.4f}")


In [ ]:
y_proba = balanced_catboost_tuned.predict_proba(X_test_cat)
proba_class_order = list(balanced_catboost_tuned.classes_)
proba_aligned = np.column_stack([y_proba[:, proba_class_order.index(c)] for c in CLASS_LABELS])

plot_calibration_curve(
    y_test, proba_aligned, CLASS_LABELS,
    title="Calibration Plots: Final Tuned CatBoost (per class)",
)

# Quantitative calibration summary, Brier score per class
print("Brier score per class:")
for i, class_name in enumerate(CLASS_LABELS):
    y_true_bin = (y_test.values == class_name).astype(int)
    brier = brier_score_loss(y_true_bin, proba_aligned[:, i])
    print(f"\t{class_name}: {brier:.4f}")

In [ ]:
# CHECKING WITH NO WEIGHTS
unweighted_catboost_tuned, unweighted_catboost_tuned_results = fit_catboost_classifier(
    "CatBoost Tuned (No Weights MLB + CAT)",
    X_train_cat,
    y_train,
    X_val_cat,
    y_val,
    cat_feature_indices=cat_feature_indices,
    class_weights=None,
    hyperparameters= nw_grid_result["params"]
)

print(f"\nFinal CatBoost model, Best iteration found: {unweighted_catboost_tuned.get_best_iteration()}")

In [ ]:
importances = unweighted_catboost_tuned.get_feature_importance()
importance_df = pd.DataFrame({
    "feature": X_train_cat.columns,
    "importance": importances,
})
importance_df["original_feature"] = importance_df["feature"].apply(
    lambda c: get_original_feature(c, CATEGORICAL_COLS)
)
importance_grouped = importance_df.groupby("original_feature")["importance"].sum().sort_values(ascending=False)

print("Feature importance (grouped by original feature):")
print(importance_grouped.to_string())

print("\nTop 15 raw features (including individual MLB sub-columns):")
print(importance_df.sort_values("importance", ascending=False).head(15).to_string(index=False))

plot_feature_importance(
    importances,
    list(X_train_cat.columns),
    title="Feature Importance: Final CatBoost Tuned Model",
    categorical_cols=CATEGORICAL_COLS,
    group_by_original_feature=True,
)

In [ ]:
permutation_results = permutation_importance(
    unweighted_catboost_tuned,
    X_test_cat,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
    scoring="f1_macro"
)

permutation_importance_df = pd.DataFrame({
    "feature": X_train_cat.columns,
    "importance_mean": permutation_results["importances_mean"],
    "importance_std": permutation_results["importances_std"]
})
permutation_importance_df["original_feature"] = permutation_importance_df["feature"].apply(
    lambda c: get_original_feature(c, CATEGORICAL_COLS)
)
permutation_grouped = permutation_importance_df.groupby("original_feature")["importance_mean"].sum().sort_values(ascending=False)

combined_importance = pd.DataFrame({
    "built_in": importance_grouped,
    "permutation": permutation_grouped
}).sort_values("built_in", ascending=False)

combined_importance_percent = combined_importance.div(combined_importance.sum()).mul(100).round(2)
print("\nCombined feature importance (built-in + permutation, grouped by original feature):")
print(combined_importance_percent.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
combined_importance_percent.iloc[::-1].plot(kind="barh", ax=ax, width=0.75)
ax.set_title("Feature Importance: Built-in vs Permutation", fontweight="bold")
ax.set_xlabel("Percentage of Total Importance")
ax.legend(["Built-in (split frequency)", "Permutation (test F1 drop)"])
plt.tight_layout()
plt.show()

print(f"\nRaw permutation importance (per encoded column, top 15):")
top_perm = permutation_importance_df.sort_values("importance_mean", ascending=False).head(15)
for _, row in top_perm.iterrows():
    print(f"\t{row['feature']}, Mean drop: {row['importance_mean']:+.4f}, Std: {row['importance_std']:.4f}")


In [ ]:
y_proba = unweighted_catboost_tuned.predict_proba(X_test_cat)
proba_class_order = list(unweighted_catboost_tuned.classes_)
proba_aligned = np.column_stack([y_proba[:, proba_class_order.index(c)] for c in CLASS_LABELS])

plot_calibration_curve(
    y_test, proba_aligned, CLASS_LABELS,
    title="Calibration Plots: Final Tuned CatBoost (per class)",
)

# Quantitative calibration summary, Brier score per class
print("Brier score per class:")
for i, class_name in enumerate(CLASS_LABELS):
    y_true_bin = (y_test.values == class_name).astype(int)
    brier = brier_score_loss(y_true_bin, proba_aligned[:, i])
    print(f"\t{class_name}: {brier:.4f}")

In [ ]:
final_catboost_params = nw_grid_result["params"]
final_catboost_class_weights = None

final_catboost_kwargs = {
    "iterations": 2000,
    "loss_function": "MultiClass",
    "random_state": RANDOM_STATE,
    "cat_features": cat_feature_indices,
    "early_stopping_rounds": 50,
    "eval_metric": "TotalF1:average=Macro",
    "verbose": False
}

final_catboost_kwargs.update(final_catboost_params)

if final_catboost_class_weights == "Balanced":
    final_catboost_kwargs["auto_class_weights"] = "Balanced"
    
final_catboost_model = CatBoostClassifier(**final_catboost_kwargs)
final_catboost_model.fit(X_train_cat, y_train, eval_set=(X_val_cat, y_val))
final_catboost_results = evaluate_classification_model(
    "Final CatBoost Tuned (No Weights MLB + CAT)",
    final_catboost_model,
    X_train_cat,
    y_train,
    X_test_cat,
    y_test,
    class_labels=CLASS_LABELS,
    cat_feature_indices=cat_feature_indices,
)

In [ ]:
# Save the model
artifacts_path = Path(ARTIFACTS_PATH)
artifacts_path.mkdir(parents=True, exist_ok=True)
final_catboost_model.save_model(str(artifacts_path / "catboost_final.cbm"))

save_encoders(mlb_encoder_final, ohe_encoder_final, artifacts_path)

classifier_model_type = {
    "model_type": "catboost",
    "model_file": "catboost_final.cbm",
    "uses_ohe": False,
    "uses_cat_features": True,
}

with open(artifacts_path / "classifier_model_type.json", "w") as f:
    json.dump(classifier_model_type, f, indent=2)
    
classifier_feature_metadata = {
    "feature_columns": list(X_train_val_cat.columns),
    "cat_feature_indices": cat_feature_indices,
    "categorical_cols": CATEGORICAL_COLS,
    "multilabel_cols": MULTILABEL_COLS,
    "class_labels": CLASS_LABELS,
    "target_col": TARGET_COL
}

with open(artifacts_path / "classifier_feature_metadata.json", "w") as f:
    json.dump(classifier_feature_metadata, f, indent=2)
    
allowed_values = {}
for col in CATEGORICAL_COLS:
    allowed_values[col] = sorted(df_train_val_cat[col].dropna().astype(str).unique().tolist())
    
for col, mlb in mlb_encoder_final.items():
    allowed_values[col] = sorted([str(value) for value in mlb.classes_])

with open(artifacts_path / "allowed_values.json", "w") as f:
    json.dump(allowed_values, f, indent=2)
    
baseline_probs = (
    y_train_val
    .value_counts(normalize=True)
    .reindex(CLASS_LABELS, fill_value=0)
    .to_dict()
)

with open(artifacts_path / "baseline_probabilities.json", "w") as f:
    json.dump(baseline_probs, f, indent=2)
    
with open(artifacts_path / "classifier_results.json", "w") as f:
    json.dump(final_catboost_results, f, indent=2)
    
print("\nFinal CatBoost model saved successfully.")